# Park 九句品質試音｜免費Colab T4
固定聲線與原文。首次執行1→2→3，核對dry-run後再啟用4，接著5逐句驗收。預設不生成。九句全接受後，才可執行另一份Remaining26 Notebook。

## 1｜安裝成功流程所需套件（GPU生成用）

In [ ]:
import sys, subprocess, os
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN']='1'
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
subprocess.check_call(['apt-get','-qq','update'])
subprocess.check_call(['apt-get','-qq','install','-y','espeak-ng','sox','libsndfile1'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','qwen-tts==0.1.1','transformers==4.57.3','accelerate==1.12.0','kokoro==0.9.4','soundfile','ipywidgets>=8,<9','huggingface-hub>=0.34,<1'])
print('套件安裝完成。若Colab提示套件已載入，請重新啟動執行階段，再從上傳格繼續。')

## 2｜上傳私人 park-voice-inputs.zip（內含核准參考及逐字稿）

In [ ]:
from google.colab import files, output as colab_output
from pathlib import Path
import uuid, zipfile, io, hashlib, importlib.util, sys
uploaded=files.upload()
if len(uploaded)!=1:raise ValueError('請只上傳 park-voice-inputs.zip')
raw=next(iter(uploaded.values()))
# Read only the known tool entry before importing code from the verified input package.
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    if sum(i.file_size for i in z.infolist())>100000000 or len(z.infolist())>300:raise ValueError('輸入ZIP過大')
    if z.namelist().count('park_tools.py')!=1:raise ValueError('輸入工具檔不符')
    tool_bytes=z.read('park_tools.py')
if hashlib.sha256(tool_bytes).hexdigest()!='30e82f59aaec5edccb78b055d42bcf65e6d36df26702477e99226d503388cb43':raise ValueError('工具程式hash不符，請使用這次提供的輸入ZIP')
spec=importlib.util.spec_from_loader('park_voice_current',loader=None)
park=importlib.util.module_from_spec(spec)
exec(compile(tool_bytes,'park_tools.py','exec'),park.__dict__)
WORK=park.unpack(raw,Path('/content')/('park_inputs_'+uuid.uuid4().hex[:8]))
park.validate_work(WORK)
colab_output.enable_custom_widget_manager()
from IPython.display import display,Markdown,Audio
OUTPUT=Path('/content')/('park_trial9_'+uuid.uuid4().hex[:8])
print('核准Rosie/Pirate主參考、逐字稿及Library共用音SHA256已核對；未生成。')

## 3｜九句dry-run，先核對角色、原文、ID與聲線

In [ ]:
dry=park.generate(WORK,OUTPUT,run=False)
rows=['|角色|聲線|lineId|英文原文|','|---|---|---|---|']
for a in dry['jobs']:rows.append('|'+a['role']+'|'+a['voiceId']+'|'+a['lineId']+'|'+a['text']+'|')
display(Markdown('\n'.join(rows)))
print('9句：Rosie3 / Pirate3 / Mermaid1 / Vampire ring2；generated=0。')
print('請核對清單後再手動啟用第4區塊；不執行26句。')

## 4｜只生成九句（手動開啟；免費T4資源可能需等候）
把RUN_GENERATION改True，CONFIRMATION填CONFIRM_PARK_NINE。重跑同一OUTPUT只續未完成句；下載ZIP後可跨執行階段恢復。

In [ ]:
RUN_GENERATION=False
CONFIRMATION=''
if RUN_GENERATION:
    print(park.generate(WORK,OUTPUT,run=True,confirmation=CONFIRMATION))
else:print('尚未生成：預設關閉。')

## 5｜真實音檔逐句驗收、自動保存、JSON匯出／載入、完整ZIP
Rosie需自然甜美童真、Pirate低沉粗獷沙啞；Mermaid成熟柔美、ring神祕低沉。檢查漏詞/語速/發音。模型只固定身份，不保證情緒品質。需要重錄只建立指定ID清單，不推論。

In [ ]:
if not (OUTPUT/'generated-manifest.json').exists():
    print('目前無生成結果；此區塊不生成。可先用第6區塊重新載入已下載結果ZIP。')
else:
    for role in ('rosie','pirate'):
        ref=park.CATALOG['characters'][role]['reference']
        print('核准主參考',role,ref['ref_text'])
        display(Audio(filename=str(WORK/'references'/ref['file'])))
    print('Pirate一致性測試（只供比較，未用作主參考）')
    display(Audio(filename=str(WORK/'references/pirate_clone_test.wav')))
    PARK_REVIEW_UI=park.show_review(OUTPUT)

## 6｜可選：重新載入已下載的Park九句結果ZIP
不需要重新生成。全新CPU驗收請改用Restore_Park_Review.ipynb，不需執行本Notebook的TTS安裝。

In [ ]:
uploaded_results=files.upload()
if len(uploaded_results)!=1:raise ValueError('請只選一個已下載Park結果ZIP')
restored=park.restore_results(next(iter(uploaded_results.values())),Path('/content'))
if park.read(restored/'generated-manifest.json')['stage']!='trial9':raise ValueError('本Notebook只恢復九句試音')
OUTPUT=restored
print('已恢復OUTPUT及原驗收紀錄：',OUTPUT)
PARK_REVIEW_UI=park.show_review(OUTPUT)
print('如要續跑未完成九句，確認同時已載入WORK輸入包後再執行第4區塊；完成音檔會依SHA略過。')

## 限制、費用及後續
無付費API，生成用免費Colab T4（可能有配額限制），Qwen模型需下載數GB；Kokoro在CPU執行。CPU恢復只需小型解碼/介面套件，無模型下載。Qwen/Kokoro模型沿Apache-2.0固定revision，參考WAV依家長核准用途，來源權利未另行獨立稽核，私人輸入包不公開。clone_test僅比較；不換聲線、不pitch假童聲。

九句原文不改，原gold事件ID繼續對應vampire ring。Bakery53/Library30不在生成清單，共用Are you happy?只copy既有WAV。下一階段26必須收到九句accepted驗收紀錄才能開啟；最終36=新35+共用1。暫存會被重置，離開前下載ZIP。此版工具有範圍/續跑/驗收測試，尚未本機GPU推論或宣稱九句音質通過。